# MEGAN 环境配置教程

## Molecule Edit Graph Attention Network (MEGAN)

> **论文**: *"Molecule Edit Graph Attention Network: Modeling Chemical Reactions as Sequences of Graph Edits"* (JCIM 2021, [arXiv:2006.15426](https://arxiv.org/abs/2006.15426))

### 方法简介

MEGAN 是一种**无模板 (template-free)** 的单步逆合成预测方法。与基于模板的方法（如 GLN）不同，MEGAN 将逆合成问题建模为**对产物分子图的序列化编辑操作**：

- **输入**: 产物分子的分子图
- **过程**: 通过图注意力网络 (GAT) 编码分子图，然后自回归地预测一系列图编辑操作（增删原子、增删改键、改变原子属性等）
- **输出**: 经过编辑后得到的反应物分子

### 核心架构
```
产物分子图 → [GAT 编码器] → 节点嵌入 → [解码器] → 编辑动作序列 → 反应物分子图
```

### 本教程目录
1. **环境配置** (本 Notebook) - 安装依赖、验证环境
2. **数据处理** - MEGAN 的图编辑数据处理流程
3. **模型展示** - MEGAN 推理原理与模型架构

### 核心依赖一览

| 依赖库 | 用途 | 推荐版本 |
|--------|------|---------|
| Python | 基础运行环境 | ≥ 3.9 |
| PyTorch | 深度学习框架 | ≥ 2.0 |
| RDKit | 化学信息学工具 | ≥ 2023.03 |
| NumPy | 数值计算 | ≥ 1.24 |
| SciPy | 稀疏矩阵存储 | ≥ 1.10 |
| pandas | 数据管理 | ≥ 2.0 |
| gin-config | 超参数管理 | ≥ 0.5 |
| tqdm | 进度条 | ≥ 4.60 |
| matplotlib | 可视化 | ≥ 3.7 |

## 1. 定义项目路径与目录结构

在整个教程中，我们使用**相对路径**来定位项目的各个组件，确保教程可移植。目录结构如下：

```
Chemical_Synthesis/                       ← 项目根目录
├── source_repos/
│   └── megan/                            ← MEGAN 源码
│       ├── src/                          ← 核心代码
│       │   ├── feat/                     ← 特征工程（图编辑动作、分子图构建）
│       │   ├── model/                    ← 模型定义（GAT编码器、解码器、Beam Search）
│       │   ├── datasets/                 ← 数据集加载
│       │   └── config.py                 ← 配置管理
│       ├── bin/                          ← 入口脚本（训练、评估、特征化）
│       ├── configs/                      ← gin 超参数配置
│       └── env.yaml                      ← 原始环境描述
├── envs/
│   └── megan_tutorial_envs/              ← MEGAN 教程环境
└── teaching_demos/
    └── 2.single_step_retro_tutorial/
        └── 2.2.template_free/
            └── 2.2.1.megan/              ← 本教程所在位置
                ├── 1_环境配置.ipynb
                ├── 2_数据处理.ipynb
                ├── 3_模型展示.ipynb
                └── data/                 ← 演示数据
```

In [1]:
from pathlib import Path
import os

# 定位项目根目录（Chemical_Synthesis）
# 从当前 notebook 位置向上回溯到项目根目录
TUTORIAL_DIR = Path(".").resolve()
PROJECT_ROOT = TUTORIAL_DIR.parents[3]  # teaching_demos -> Chemical_Synthesis

# 核心路径定义（全部使用相对路径）
SOURCE_REPOS_DIR = PROJECT_ROOT / "source_repos"
MEGAN_SOURCE_DIR = SOURCE_REPOS_DIR / "megan"
ENVS_DIR = PROJECT_ROOT / "envs"
MEGAN_ENV_DIR = ENVS_DIR / "megan_tutorial_envs"
DATA_DIR = TUTORIAL_DIR / "data"

print(f"项目根目录:   {PROJECT_ROOT}")
print(f"MEGAN 源码:   {MEGAN_SOURCE_DIR}")
print(f"环境目录:     {MEGAN_ENV_DIR}")
print(f"教程数据目录: {DATA_DIR}")
print(f"\n目录存在性检查:")
for name, path in [("项目根目录", PROJECT_ROOT), ("源码目录", MEGAN_SOURCE_DIR), 
                    ("数据目录", DATA_DIR)]:
    print(f"  {name}: {'✓ 存在' if path.exists() else '✗ 不存在'}")

项目根目录:   /home/xiaoruiwang/backup_data/ubuntu_data/other_work/GNN_AIDD/Chemical_Synthesis
MEGAN 源码:   /home/xiaoruiwang/backup_data/ubuntu_data/other_work/GNN_AIDD/Chemical_Synthesis/source_repos/megan
环境目录:     /home/xiaoruiwang/backup_data/ubuntu_data/other_work/GNN_AIDD/Chemical_Synthesis/envs/megan_tutorial_envs
教程数据目录: /home/xiaoruiwang/backup_data/ubuntu_data/other_work/GNN_AIDD/Chemical_Synthesis/teaching_demos/2.single_step_retro_tutorial/2.2.template_free/2.2.1.megan/data

目录存在性检查:
  项目根目录: ✓ 存在
  源码目录: ✓ 存在
  数据目录: ✓ 存在


## 2. 克隆 MEGAN 源码仓库

MEGAN 的源码托管在 GitHub 上。如果还未克隆，执行以下命令将其下载到 `source_repos/megan` 目录。

> **注意**: MEGAN 原始代码基于 Python 3.6 + PyTorch 1.3.1，本教程将适配到**现代版本**（Python 3.9+, PyTorch 2.x），核心算法逻辑保持不变。

In [2]:
import subprocess

if MEGAN_SOURCE_DIR.exists():
    print(f"✓ MEGAN 源码已存在于: {MEGAN_SOURCE_DIR}")
else:
    print("正在克隆 MEGAN 仓库...")
    subprocess.run(
        ["git", "clone", "https://github.com/molecule-one/megan.git", str(MEGAN_SOURCE_DIR)],
        check=True
    )
    print(f"✓ 克隆完成: {MEGAN_SOURCE_DIR}")

# 展示 MEGAN 源码目录结构
print("\n=== MEGAN 源码目录结构 ===")
for item in sorted(MEGAN_SOURCE_DIR.iterdir()):
    prefix = "📁" if item.is_dir() else "📄"
    print(f"  {prefix} {item.name}/") if item.is_dir() else print(f"  {prefix} {item.name}")
    if item.is_dir() and item.name in ("src", "bin", "configs"):
        for sub in sorted(item.iterdir()):
            sub_prefix = "📁" if sub.is_dir() else "📄"
            print(f"      {sub_prefix} {sub.name}")

✓ MEGAN 源码已存在于: /home/xiaoruiwang/backup_data/ubuntu_data/other_work/GNN_AIDD/Chemical_Synthesis/source_repos/megan

=== MEGAN 源码目录结构 ===
  📁 .git/
  📄 LICENSE
  📄 README.md
  📁 bin/
      📄 __init__.py
      📄 acquire.py
      📄 eval.py
      📄 featurize.py
      📄 train.py
  📁 configs/
      📄 uspto_50k.gin
      📄 uspto_50k_rt.gin
      📄 uspto_full.gin
      📄 uspto_mit.gin
      📄 uspto_mit_sep.gin
  📁 data/
  📄 env.sh
  📄 env.yaml
  📁 logs/
  📁 models/
  📁 src/
      📄 __init__.py
      📄 config.py
      📁 datasets
      📁 feat
      📄 get_instances.py
      📁 model
      📁 split
      📁 utils


## 3. 创建虚拟环境

我们在 `envs/megan_tutorial_envs` 目录下创建一个独立的 Conda 环境。MEGAN 原始环境基于 Python 3.6，本教程适配到 **Python 3.9**，兼容性最佳。

### 安装步骤概览
1. 创建 Conda 环境（Python 3.9）
2. 安装 PyTorch 2.x（根据 CUDA 版本选择）
3. 安装 RDKit（通过 conda-forge）
4. 安装 MEGAN 项目依赖

In [3]:
%%bash
# ============================================================
# 步骤 1: 创建 Conda 环境
# ============================================================
# 获取项目根目录（相对于本 notebook 的位置）
PROJECT_ROOT="$(cd ../../../.. && pwd)"
ENV_DIR="${PROJECT_ROOT}/envs/megan_tutorial_envs"

echo "=== 创建 MEGAN 教程环境 ==="
echo "环境路径: ${ENV_DIR}"

# 如果环境已存在则跳过
if [ -d "${ENV_DIR}" ] && [ -f "${ENV_DIR}/bin/python" ]; then
    echo "✓ 环境已存在，跳过创建"
    ${ENV_DIR}/bin/python --version
else
    echo "正在创建 Conda 环境..."
    conda create --prefix "${ENV_DIR}" python=3.9 -y -q
    echo "✓ 环境创建完成"
fi

=== 创建 MEGAN 教程环境 ===
环境路径: /home/xiaoruiwang/backup_data/ubuntu_data/other_work/GNN_AIDD/Chemical_Synthesis/envs/megan_tutorial_envs
正在创建 Conda 环境...
Channels:
 - defaults
Platform: linux-64
Solving environment: ...working... done

## Package Plan ##

  environment location: /home/xiaoruiwang/backup_data/ubuntu_data/other_work/GNN_AIDD/Chemical_Synthesis/envs/megan_tutorial_envs

  added / updated specs:
    - python=3.9


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    python-3.9.25              |       h0dcde21_1        23.0 MB  defaults
    setuptools-80.9.0          |   py39h06a4308_0         1.4 MB  defaults
    wheel-0.45.1               |   py39h06a4308_0         114 KB  defaults
    ------------------------------------------------------------
                                           Total:        24.6 MB

The following NEW packages will be INSTALLED:

  _libgcc_mutex      anac

## 4. 安装 PyTorch

MEGAN 的原始版本使用 PyTorch 1.3.1，本教程升级到 **PyTorch 2.x**，核心 API 完全兼容。

> **提示**: 根据你的 CUDA 版本选择对应的 PyTorch 安装命令。如果没有 GPU，可以安装 CPU 版本。

In [4]:
%%bash
PROJECT_ROOT="$(cd ../../../.. && pwd)"
ENV_DIR="${PROJECT_ROOT}/envs/megan_tutorial_envs"
PIP="${ENV_DIR}/bin/pip"

echo "=== 安装 PyTorch ==="

# 检测 CUDA 版本并安装对应的 PyTorch
if command -v nvidia-smi &> /dev/null; then
    CUDA_VERSION=$(nvidia-smi | grep "CUDA Version" | awk '{print $9}')
    echo "检测到 CUDA 版本: ${CUDA_VERSION}"
    # 安装 CUDA 版 PyTorch (使用 PyTorch 官方源)
    ${PIP} install torch torchvision --index-url https://download.pytorch.org/whl/cu121 -q 2>&1 | tail -3
else
    echo "未检测到 CUDA，安装 CPU 版本"
    ${PIP} install torch torchvision --index-url https://download.pytorch.org/whl/cpu -q 2>&1 | tail -3
fi

echo ""
echo "=== PyTorch 安装验证 ==="
${ENV_DIR}/bin/python -c "
import torch
print(f'PyTorch 版本: {torch.__version__}')
print(f'CUDA 可用: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'CUDA 版本: {torch.version.cuda}')
    print(f'GPU 设备: {torch.cuda.get_device_name(0)}')
"

=== 安装 PyTorch ===
检测到 CUDA 版本: 12.4
ndfes 3.0 requires matplotlib, which is not installed.
ndfes 3.0 requires scipy, which is not installed.

=== PyTorch 安装验证 ===
PyTorch 版本: 2.5.1+cu121
CUDA 可用: True
CUDA 版本: 12.1
GPU 设备: NVIDIA GeForce RTX 3060


## 5. 安装 RDKit 与化学信息学工具

RDKit 是化学信息学的核心库，MEGAN 使用它来：
- 解析 SMILES 字符串为分子对象
- 获取原子/键属性
- 操作分子图（添加/删除原子和键）
- 处理原子映射编号 (atom mapping)

In [5]:
%%bash
PROJECT_ROOT="$(cd ../../../.. && pwd)"
ENV_DIR="${PROJECT_ROOT}/envs/megan_tutorial_envs"
CONDA_BIN="$(which conda)"

echo "=== 安装 RDKit ==="
# 使用 conda-forge 安装 RDKit（最可靠的方式）
${CONDA_BIN} install --prefix "${ENV_DIR}" -c conda-forge rdkit -y -q 2>&1 | tail -5

echo ""
echo "=== RDKit 安装验证 ==="
${ENV_DIR}/bin/python -c "
from rdkit import Chem, rdBase
print(f'RDKit 版本: {rdBase.rdkitVersion}')
mol = Chem.MolFromSmiles('c1ccccc1')
print(f'苯分子 SMILES: {Chem.MolToSmiles(mol)}')
print(f'原子数: {mol.GetNumAtoms()}, 键数: {mol.GetNumBonds()}')
print('✓ RDKit 功能正常')
"

=== 安装 RDKit ===


Preparing transaction: ...working... done
Verifying transaction: ...working... done
Executing transaction: ...working... done

=== RDKit 安装验证 ===
RDKit 版本: 2025.03.5
苯分子 SMILES: c1ccccc1
原子数: 6, 键数: 6
✓ RDKit 功能正常


## 6. 安装 MEGAN 项目依赖

MEGAN 使用以下额外依赖：
- **gin-config**: Google 的超参数配置管理框架
- **scipy**: 稀疏矩阵存储（用于保存图特征数据）
- **pandas**: 数据管理
- **tqdm**: 进度条
- **matplotlib**: 可视化

> **注意**: MEGAN 原始代码依赖 TensorFlow 仅用于 TensorBoard 日志，本教程不需要。

In [6]:
%%bash
PROJECT_ROOT="$(cd ../../../.. && pwd)"
ENV_DIR="${PROJECT_ROOT}/envs/megan_tutorial_envs"
PIP="${ENV_DIR}/bin/pip"

echo "=== 安装 MEGAN 项目依赖 ==="
${PIP} install \
    numpy pandas scipy matplotlib tqdm \
    gin-config ipykernel \
    -q 2>&1 | tail -5

echo ""
echo "=== 注册 Jupyter Kernel ==="
${ENV_DIR}/bin/python -m ipykernel install --user \
    --name megan_tutorial \
    --display-name "MEGAN Tutorial (Python 3.9)" 2>&1

echo ""
echo "✓ 所有依赖安装完成"
echo "提示: 请将 Jupyter Kernel 切换到 'MEGAN Tutorial (Python 3.9)' 后再运行后续 Notebook"

=== 安装 MEGAN 项目依赖 ===

=== 注册 Jupyter Kernel ===
Installed kernelspec megan_tutorial in /home/xiaoruiwang/.local/share/jupyter/kernels/megan_tutorial

✓ 所有依赖安装完成
提示: 请将 Jupyter Kernel 切换到 'MEGAN Tutorial (Python 3.9)' 后再运行后续 Notebook


## 7. 环境验证

> ⚠️ **切换内核**: 请先将 Jupyter Kernel 切换到 `MEGAN Tutorial (Python 3.9)` 再运行以下验证代码。

### 7.1 基础库导入与版本检查

In [1]:
# ============================================================
# 环境验证清单
# ============================================================
import sys
print(f"Python 版本: {sys.version}")
print(f"Python 路径: {sys.executable}\n")

libraries = {}

# PyTorch
try:
    import torch
    libraries['PyTorch'] = torch.__version__
    if torch.cuda.is_available():
        libraries['CUDA'] = torch.version.cuda
        libraries['GPU'] = torch.cuda.get_device_name(0)
except ImportError:
    libraries['PyTorch'] = '✗ 未安装'

# RDKit
try:
    from rdkit import Chem, rdBase
    libraries['RDKit'] = rdBase.rdkitVersion
except ImportError:
    libraries['RDKit'] = '✗ 未安装'

# NumPy
try:
    import numpy as np
    libraries['NumPy'] = np.__version__
except ImportError:
    libraries['NumPy'] = '✗ 未安装'

# SciPy
try:
    import scipy
    libraries['SciPy'] = scipy.__version__
except ImportError:
    libraries['SciPy'] = '✗ 未安装'

# pandas
try:
    import pandas as pd
    libraries['pandas'] = pd.__version__
except ImportError:
    libraries['pandas'] = '✗ 未安装'

# gin-config
try:
    import gin
    libraries['gin-config'] = gin.__version__ if hasattr(gin, '__version__') else '已安装'
except ImportError:
    libraries['gin-config'] = '✗ 未安装'

# matplotlib
try:
    import matplotlib
    libraries['matplotlib'] = matplotlib.__version__
except ImportError:
    libraries['matplotlib'] = '✗ 未安装'

# tqdm
try:
    import tqdm
    libraries['tqdm'] = tqdm.__version__
except ImportError:
    libraries['tqdm'] = '✗ 未安装'

# 打印结果
print("=" * 45)
print(f"{'库名':<15} {'版本':<25}")
print("=" * 45)
for lib, ver in libraries.items():
    status = "✓" if "未安装" not in str(ver) else "✗"
    print(f"{status} {lib:<13} {ver}")
print("=" * 45)

Python 版本: 3.9.25 (main, Nov  3 2025, 22:33:05) 
[GCC 11.2.0]
Python 路径: /home/xiaoruiwang/backup_data/ubuntu_data/other_work/GNN_AIDD/Chemical_Synthesis/envs/megan_tutorial_envs/bin/python

库名              版本                       
✓ PyTorch       2.5.1+cu121
✓ CUDA          12.1
✓ GPU           NVIDIA GeForce RTX 3060
✓ RDKit         2025.03.5
✓ NumPy         2.0.2
✓ SciPy         1.13.1
✓ pandas        2.3.1
✓ gin-config    已安装
✓ matplotlib    3.9.4
✓ tqdm          4.67.3


### 7.2 RDKit 基础功能验证

MEGAN 大量使用 RDKit 的以下功能：
- **SMILES 解析**: 将 SMILES 字符串转换为分子对象
- **原子映射**: 追踪反应中原子的对应关系
- **分子编辑**: 使用 `RWMol` 对分子进行增删原子/键操作
- **特征提取**: 获取原子类型、电荷、芳香性等属性

In [2]:
from rdkit import Chem
from rdkit.Chem import AllChem, Draw, rdchem
import numpy as np

# ============================================================
# 验证 1: SMILES 解析与原子映射
# ============================================================
# 一个带有原子映射的反应示例 (阿司匹林水解)
product_smi = "[CH3:1][C:2](=[O:3])[O:4][c:5]1[cH:6][cH:7][cH:8][cH:9][c:10]1[C:11](=[O:12])[OH:13]"
prod_mol = Chem.MolFromSmiles(product_smi)

print("=== SMILES 解析与原子映射 ===")
print(f"带映射的 SMILES: {product_smi}")
print(f"标准化 SMILES:   {Chem.MolToSmiles(prod_mol)}")
print(f"原子数: {prod_mol.GetNumAtoms()}, 键数: {prod_mol.GetNumBonds()}")
print("\n原子信息:")
for atom in prod_mol.GetAtoms():
    print(f"  原子 {atom.GetIdx()}: {atom.GetSymbol()} "
          f"(映射号={atom.GetAtomMapNum()}, "
          f"电荷={atom.GetFormalCharge()}, "
          f"芳香性={'是' if atom.GetIsAromatic() else '否'})")

# ============================================================
# 验证 2: 分子编辑 (RWMol) - MEGAN 的核心操作
# ============================================================
print("\n=== RWMol 分子编辑演示 ===")
# 创建一个可编辑的分子 (乙醇)
rw_mol = Chem.RWMol(Chem.MolFromSmiles("[CH3:1][OH:2]"))
print(f"初始分子: {Chem.MolToSmiles(rw_mol)}")

# 添加新原子（模拟 MEGAN 的 AddAtomAction）
new_atom = Chem.Atom(6)  # 碳原子
new_atom.SetAtomMapNum(3)
idx = rw_mol.AddAtom(new_atom)
rw_mol.AddBond(0, idx, Chem.rdchem.BondType.SINGLE)
print(f"添加碳原子后: {Chem.MolToSmiles(rw_mol)}")

# 修改键类型（模拟 MEGAN 的 BondEditAction）
bond = rw_mol.GetBondBetweenAtoms(0, idx)
bond.SetBondType(Chem.rdchem.BondType.DOUBLE)
print(f"改为双键后:   {Chem.MolToSmiles(rw_mol)}")

print("\n✓ RDKit 分子编辑功能正常")

=== SMILES 解析与原子映射 ===
带映射的 SMILES: [CH3:1][C:2](=[O:3])[O:4][c:5]1[cH:6][cH:7][cH:8][cH:9][c:10]1[C:11](=[O:12])[OH:13]
标准化 SMILES:   [CH3:1][C:2](=[O:3])[O:4][c:5]1[cH:6][cH:7][cH:8][cH:9][c:10]1[C:11](=[O:12])[OH:13]
原子数: 13, 键数: 13

原子信息:
  原子 0: C (映射号=1, 电荷=0, 芳香性=否)
  原子 1: C (映射号=2, 电荷=0, 芳香性=否)
  原子 2: O (映射号=3, 电荷=0, 芳香性=否)
  原子 3: O (映射号=4, 电荷=0, 芳香性=否)
  原子 4: C (映射号=5, 电荷=0, 芳香性=是)
  原子 5: C (映射号=6, 电荷=0, 芳香性=是)
  原子 6: C (映射号=7, 电荷=0, 芳香性=是)
  原子 7: C (映射号=8, 电荷=0, 芳香性=是)
  原子 8: C (映射号=9, 电荷=0, 芳香性=是)
  原子 9: C (映射号=10, 电荷=0, 芳香性=是)
  原子 10: C (映射号=11, 电荷=0, 芳香性=否)
  原子 11: O (映射号=12, 电荷=0, 芳香性=否)
  原子 12: O (映射号=13, 电荷=0, 芳香性=否)

=== RWMol 分子编辑演示 ===
初始分子: [CH3:1][OH:2]
添加碳原子后: [CH3:1]([OH:2])[CH3:3]
改为双键后:   [CH3:1]([OH:2])=[CH2:3]

✓ RDKit 分子编辑功能正常


### 7.3 PyTorch 图操作验证

MEGAN 使用 PyTorch 构建图注意力网络。以下验证 PyTorch 的基础张量操作和简单的注意力计算：

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# ============================================================
# 简单的图注意力计算演示（MEGAN 使用的核心操作）
# ============================================================
print("=== 图注意力计算演示 ===\n")

# 模拟一个 3 节点的分子图（不含超节点）
n_nodes = 3
hidden_dim = 4

# 节点特征: (n_nodes, hidden_dim)
node_feats = torch.randn(n_nodes, hidden_dim)
print(f"节点特征形状: {node_feats.shape}")

# 邻接矩阵: (n_nodes, n_nodes)
# 0-1-2 链式连接
adj = torch.zeros(n_nodes, n_nodes)
adj[0, 1] = adj[1, 0] = 1  # 0-1 连接
adj[1, 2] = adj[2, 1] = 1  # 1-2 连接
print(f"邻接矩阵:\n{adj}")

# 注意力计算 (MEGAN 风格)
# 对每对节点 (i, j) 计算注意力分数
att_proj = nn.Linear(hidden_dim, hidden_dim // 2)
att_final = nn.Linear(hidden_dim, 1)

x_att = torch.relu(att_proj(node_feats))  # (n_nodes, hidden_dim//2)

# 构造节点对特征：拼接 node_i 和 node_j 的特征
x_rows = x_att.unsqueeze(0).expand(n_nodes, -1, -1)  # (n, n, d)
x_cols = x_att.unsqueeze(1).expand(-1, n_nodes, -1)  # (n, n, d)
pair_feats = torch.cat([x_rows, x_cols], dim=-1)       # (n, n, 2d)

# 计算注意力权重
att_scores = att_final(pair_feats).squeeze(-1)          # (n, n)
# 掩码：只关注有边连接的节点
mask = (1.0 - adj) * -1e9
att_weights = torch.softmax(att_scores + mask, dim=-1) * adj

print(f"\n注意力权重矩阵:\n{att_weights.detach().numpy().round(3)}")

# 消息传递：加权聚合邻居特征
updated_feats = torch.bmm(att_weights.unsqueeze(0), node_feats.unsqueeze(0)).squeeze(0)
print(f"\n更新后节点特征形状: {updated_feats.shape}")
print("\n✓ PyTorch 图注意力计算正常")

=== 图注意力计算演示 ===

节点特征形状: torch.Size([3, 4])
邻接矩阵:
tensor([[0., 1., 0.],
        [1., 0., 1.],
        [0., 1., 0.]])

注意力权重矩阵:
[[0.   1.   0.  ]
 [0.49 0.   0.51]
 [0.   1.   0.  ]]

更新后节点特征形状: torch.Size([3, 4])

✓ PyTorch 图注意力计算正常


### 7.4 MEGAN 源码结构概览

MEGAN 源码组织如下，对应本教程的三个核心部分：

In [4]:
from pathlib import Path

PROJECT_ROOT = Path(".").resolve().parents[3]
MEGAN_SOURCE_DIR = PROJECT_ROOT / "source_repos" / "megan"

# MEGAN 模块说明
module_descriptions = {
    "src/feat/": "📊 特征工程 - 图编辑动作定义、分子图构建、动作序列生成",
    "src/feat/reaction_actions.py": "   ├── 反应动作类: AtomEditAction, BondEditAction, AddAtomAction, AddRingAction, StopAction",
    "src/feat/megan_graph.py":      "   ├── 训练样本特征化器: MeganTrainingSamplesFeaturizer",
    "src/feat/mol_graph.py":        "   ├── 分子图构建: get_graph() - 含超节点(supernode)的图表示",
    "src/feat/graph_features.py":   "   ├── 原子/键特征: 属性值的 one-hot 编码映射",
    "src/feat/featurize.py":        "   └── 核心逻辑: ReactionSampleGenerator - 生成图编辑训练样本",
    "src/model/": "🧠 模型架构 - GAT 编码器 + 动作预测解码器",
    "src/model/megan.py":           "   ├── 主模型: Megan - 编码器+解码器的组合，状态化/无状态化生成",
    "src/model/megan_modules/encoder.py": "   ├── 编码器: MeganEncoder - 多层 Multi-Head GAT",
    "src/model/megan_modules/decoder.py": "   ├── 解码器: MeganDecoder - 原子动作头 + 键动作头",
    "src/model/graph/gat.py":       "   ├── GAT层: MultiHeadGraphConvLayer - 多头图注意力卷积",
    "src/model/beam_search.py":     "   └── 推理: beam_search() - 束搜索解码",
    "src/datasets/": "📁 数据集 - USPTO-50K/MIT/FULL 数据加载",
    "src/config.py": "⚙️  配置管理 - 数据集、特征化器、分割方案的注册",
    "bin/": "🔧 入口脚本 - acquire.py(下载), featurize.py(特征化), train.py(训练), eval.py(评估)",
}

print("=== MEGAN 源码模块说明 ===\n")
for module, desc in module_descriptions.items():
    path = MEGAN_SOURCE_DIR / module.rstrip('/')
    exists = "✓" if path.exists() else "✗"
    print(f"  {exists} {module:<45} {desc}")

=== MEGAN 源码模块说明 ===

  ✓ src/feat/                                     📊 特征工程 - 图编辑动作定义、分子图构建、动作序列生成
  ✓ src/feat/reaction_actions.py                     ├── 反应动作类: AtomEditAction, BondEditAction, AddAtomAction, AddRingAction, StopAction
  ✓ src/feat/megan_graph.py                          ├── 训练样本特征化器: MeganTrainingSamplesFeaturizer
  ✓ src/feat/mol_graph.py                            ├── 分子图构建: get_graph() - 含超节点(supernode)的图表示
  ✓ src/feat/graph_features.py                       ├── 原子/键特征: 属性值的 one-hot 编码映射
  ✓ src/feat/featurize.py                            └── 核心逻辑: ReactionSampleGenerator - 生成图编辑训练样本
  ✓ src/model/                                    🧠 模型架构 - GAT 编码器 + 动作预测解码器
  ✓ src/model/megan.py                               ├── 主模型: Megan - 编码器+解码器的组合，状态化/无状态化生成
  ✓ src/model/megan_modules/encoder.py               ├── 编码器: MeganEncoder - 多层 Multi-Head GAT
  ✓ src/model/megan_modules/decoder.py               ├── 解码器: MeganDecoder - 原子动作头 + 键动作头
  ✓ src/model/graph/gat.py   

## 8. MEGAN 核心概念预览

### 与模板方法的对比

| 特性 | 基于模板 (如 GLN) | 无模板 (MEGAN) |
|------|-------------------|---------------|
| 反应表示 | 反应模板 (SMARTS) | 图编辑序列 |
| 预测方式 | 选择最佳模板并应用 | 逐步编辑产物分子图 |
| 处理未见反应 | 仅能处理训练集中出现的模板 | 可生成任意结构变换 |
| 输出确定性 | 模板应用结果确定 | 多步编辑可产生多种路径 |

### MEGAN 的 6 种图编辑动作

MEGAN 定义了以下 6 种基本图编辑操作，覆盖了逆合成中所有可能的分子变换：

| 动作 | 类名 | 说明 | 示例 |
|------|------|------|------|
| 编辑原子 | `AtomEditAction` | 修改原子属性（电荷、手性、芳香性等） | 改变形式电荷 |
| 添加原子 | `AddAtomAction` | 在现有原子旁添加新原子和键 | 添加甲基 |
| 编辑键 | `BondEditAction` | 修改/添加/删除化学键 | 断裂 C-C 键 |
| 添加环 | `AddRingAction` | 添加芳香环结构（如苯环） | 添加苯环 |
| 停止 | `StopAction` | 结束编辑序列 | 生成完成 |

### 工作流程概览
```
训练阶段:
  反应数据 (product → reactants)
      ↓ 提取图编辑序列
  (产物图, 动作序列) 训练样本对
      ↓ GAT 编码器 + 动作预测解码器
  训练模型参数

推理阶段:
  产物 SMILES
      ↓ 构建分子图
  GAT 编码
      ↓ 自回归解码 (Beam Search)
  反应物 SMILES 候选列表
```

---

### 下一步

环境配置完成！请继续学习：
- **Notebook 2 (数据处理)**: 深入了解 MEGAN 如何将反应数据转化为图编辑训练样本
- **Notebook 3 (模型展示)**: 详细解析 MEGAN 的 GAT 编码器和动作预测解码器